# MyTravels — Preview Runbook

Runs the full MyTravels stack via Docker Compose. No local SDK or build tools required.

| Service | Purpose | Port(s) |
|---------|---------|--------|
| PostgreSQL | Primary database | 5432 |
| RabbitMQ | Message broker | 5672 · 15672 (UI) |
| MinIO | Object storage | 9000 (API) · 9090 (Console) |
| API | REST API | 5101 |
| Messaging | Background worker | 5102 |

> Run each cell in order.

## Prerequisites

| Tool | Purpose | Install |
|------|---------|---------|
| Rancher Desktop | Docker engine + `docker compose` | `brew install --cask rancher` |
| JupyterLab | Run this notebook | `brew install jupyterlab` |

Once Rancher Desktop is installed, open it and ensure the container engine is running before continuing.

- A `.env` file must exist in this directory (copy from `.env.example` if one exists)

---
## Build and Push Docker Images

Build all service images and push them to the container registry. This uses the `docker-compose.build.yml` file which defines the build targets and registry destination for each service.

> **Before running:** ensure you are logged in to your container registry (`docker login`) and that `docker-compose.build.yml` is configured with the correct image names and registry.

In [ ]:
%%bash
docker compose -f docker-compose.build.yml build --push

---
## 1. Load environment variables

In [7]:
from pathlib import Path
import os

for line in Path(".env").read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        key, _, value = line.partition("=")
        os.environ[key.strip()] = value.strip()

print("Loaded environment from .env")

Loaded environment from .env


---
## 2. Start the stack

In [8]:
%%bash
docker compose up -d
echo "Stack started."

time="2026-05-17T12:58:13+02:00" level=warning msg="No services to build"
 Network 0-preview_default Creating 
 Network 0-preview_default Created 
 Volume 0-preview_mqdata Creating 
 Volume 0-preview_mqdata Created 
 Volume 0-preview_mqconfig Creating 
 Volume 0-preview_mqconfig Created 
 Volume 0-preview_pgdata Creating 
 Volume 0-preview_pgdata Created 
 Volume 0-preview_minio-data Creating 
 Volume 0-preview_minio-data Created 
 Volume 0-preview_minio-config Creating 
 Volume 0-preview_minio-config Created 
 Container minio Creating 
 Container mytravels-rabbitmq Creating 
 Container mytravels-postgres Creating 
 Container mytravels-rabbitmq Created 
 Container mytravels-postgres Created 
 Container cleanup-migrations Creating 
 Container 0-preview-migrate-core-db-1 Creating 
 Container minio Created 
 Container 0-preview-migrate-core-db-1 Created 
 Container cleanup-migrations Created 
 Container mytravels-messaging Creating 
 Container mytravels-api Creating 
 Container mytravels-

Stack started.


---
## 3. Check service health

In [9]:
%%bash
docker compose ps

NAME                  IMAGE                                       COMMAND                  SERVICE     CREATED         STATUS                   PORTS
minio                 quay.io/minio/minio                         "/usr/bin/docker-ent…"   minio       2 minutes ago   Up 2 minutes (healthy)   0.0.0.0:9000->9000/tcp, [::]:9000->9000/tcp, 0.0.0.0:9090->9090/tcp, [::]:9090->9090/tcp
mytravels-api         tshepontlhokoa/mytravels-api:v1.0.2         "dotnet mytravels.ap…"   api         2 minutes ago   Up About a minute        0.0.0.0:5101->5101/tcp, [::]:5101->5101/tcp
mytravels-messaging   tshepontlhokoa/mytravels-messaging:v1.0.4   "dotnet mytravels.me…"   messaging   2 minutes ago   Up About a minute        0.0.0.0:5102->5102/tcp, [::]:5102->5102/tcp
mytravels-postgres    postgres:17.6-alpine                        "docker-entrypoint.s…"   postgres    2 minutes ago   Up 2 minutes (healthy)   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp
mytravels-rabbitmq    rabbitmq:3-management          

**Service UIs:**
- RabbitMQ Management: http://localhost:15672
- MinIO Console: http://localhost:9090
- API: http://localhost:5101
- Messaging: http://localhost:5102

---
## 4. View logs (optional)

In [10]:
%%bash
echo "=== minio ===" && docker compose logs --tail=20 minio
echo "=== rabbitmq ===" && docker compose logs --tail=20 rabbitmq
echo "=== postgres ===" && docker compose logs --tail=20 postgres
echo "=== cleanup-migrations ===" && docker compose logs --tail=20 cleanup-migrations
echo "=== migrate-core-db ===" && docker compose logs --tail=20 migrate-core-db
echo "=== api ===" && docker compose logs --tail=20 api
echo "=== messaging ===" && docker compose logs --tail=20 messaging

=== minio ===
minio  | INFO: Formatting 1st pool, 1 set(s), 1 drives per set.
minio  | INFO: WARNING: Host local has more than 0 drives of set. A host failure will result in data becoming unavailable.
minio  | MinIO Object Storage Server
minio  | Copyright: 2015-2026 MinIO, Inc.
minio  | License: GNU AGPLv3 - https://www.gnu.org/licenses/agpl-3.0.html
minio  | Version: RELEASE.2025-09-07T16-13-09Z (go1.24.6 linux/arm64)
minio  | 
minio  | API: http://172.22.0.3:9000  http://127.0.0.1:9000 
minio  | WebUI: http://172.22.0.3:9090 http://127.0.0.1:9090  
minio  | 
minio  | Docs: https://docs.min.io
=== rabbitmq ===
mytravels-rabbitmq  | 2026-05-17 10:58:16.465409+00:00 [info] <0.742.0> Management plugin: HTTP (non-TLS) listener started on port 15672
mytravels-rabbitmq  | 2026-05-17 10:58:16.465494+00:00 [info] <0.772.0> Statistics database started.
mytravels-rabbitmq  | 2026-05-17 10:58:16.465530+00:00 [info] <0.771.0> Starting worker pool 'management_worker_pool' with 3 processes in it
m

In [11]:
%%bash
# Stop and remove containers AND volumes — wipes all data
docker compose down --volumes
echo "Containers and volumes removed."

 Container mytravels-messaging Stopping 
 Container cleanup-migrations Stopping 
 Container minio Stopping 
 Container mytravels-api Stopping 
 Container cleanup-migrations Stopped 
 Container cleanup-migrations Removing 
 Container cleanup-migrations Removed 
 Container minio Stopped 
 Container minio Removing 
 Container minio Removed 
 Container mytravels-messaging Stopped 
 Container mytravels-messaging Removing 
 Container mytravels-messaging Removed 
 Container mytravels-api Stopped 
 Container mytravels-api Removing 
 Container mytravels-api Removed 
 Container mytravels-rabbitmq Stopping 
 Container 0-preview-migrate-core-db-1 Stopping 
 Container 0-preview-migrate-core-db-1 Stopped 
 Container 0-preview-migrate-core-db-1 Removing 
 Container 0-preview-migrate-core-db-1 Removed 
 Container mytravels-postgres Stopping 
 Container mytravels-postgres Stopped 
 Container mytravels-postgres Removing 
 Container mytravels-postgres Removed 
 Container mytravels-rabbitmq Stopped 
 Cont

Containers and volumes removed.
